In [1]:
"""
NSGA-III Multi-Objective Hyperparameter Optimization for MLP on MNIST
======================================================================
Reproduces Table 3 from the paper:
  "Multi-objective Hyperparameter Optimization of Deep Learning Models with NSGA-III"

Target (Table 3):
  OPT=Adagrad, LR=0.1, EP=50, BS=32
  Accuracy=0.9835, F1=0.9834, Parameters=101,770, Weighted Sum=2.9669

Objectives (all maximized via negation where needed):
  1. Maximize Accuracy
  2. Maximize F1-score
  3. Maximize inversed_normalized_params  (= minimize parameter count)

Weighted Sum = F1_score * 1 + Accuracy * 1 + inversed_normalized_params * 1
"""

# ── Suppress TF/Keras noise ──────────────────────────────────────────────────
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from itertools import combinations
from collections import Counter
from scipy.linalg import LinAlgError
from scipy.spatial.distance import cdist

import tensorflow as tf
tf.get_logger().setLevel("ERROR")

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score

# ════════════════════════════════════════════════════════════════════════════
# 1.  HYPERPARAMETER SEARCH SPACE  (MLP only – Table 1)
# ════════════════════════════════════════════════════════════════════════════
# Index → actual value mappings
OPT_MAP  = {0: "SGD", 1: "RMSprop", 2: "Adagrad",
            3: "Adadelta", 4: "Adam", 5: "Adamax", 6: "Nadam"}
ACT_MAP  = {0: "relu", 1: "elu"}
LR_MAP   = {0: 1.0, 1: 0.1, 2: 0.01, 3: 0.001, 4: 0.0001}
EP_MAP   = {0: 10,  1: 20,  2: 30,   3: 40,    4: 50}
BS_MAP   = {0: 32,  1: 64,  2: 128,  3: 256,   4: 512}

# Encoded search-space bounds [OPT, ACT, LR, EP, BS]
LB = np.array([0, 0, 0, 0, 0], dtype=float)
UB = np.array([6, 1, 4, 4, 4], dtype=float)
NVAR = len(LB)

# ════════════════════════════════════════════════════════════════════════════
# 2.  DATA LOADING
# ════════════════════════════════════════════════════════════════════════════
def load_mnist():
    """Load & preprocess MNIST exactly as the paper describes."""
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
    x_train = x_train.reshape(-1, 28 * 28).astype("float32") / 255.0
    x_test  = x_test.reshape(-1, 28 * 28).astype("float32") / 255.0
    return x_train, y_train, x_test, y_test

X_TRAIN, Y_TRAIN, X_TEST, Y_TEST = load_mnist()

# ════════════════════════════════════════════════════════════════════════════
# 3.  MLP BUILDER & EVALUATOR
# ════════════════════════════════════════════════════════════════════════════
# Maximum possible params for MLP on MNIST (used for normalisation).
# MLP: Flatten(784) → Dense(128, act) → Dense(10, softmax)
# 784*128 + 128 + 128*10 + 10 = 100352 + 128 + 1280 + 10 = 101,770
MAX_PARAMS = 101_770   # fixed for MLP – all configs share the same architecture


def decode(individual):
    """Map a continuous encoded individual → discrete hyperparameters."""
    ind = np.round(individual).astype(int)
    ind = np.clip(ind, LB.astype(int), UB.astype(int))
    opt = OPT_MAP[ind[0]]
    act = ACT_MAP[ind[1]]
    lr  = LR_MAP[ind[2]]
    ep  = EP_MAP[ind[3]]
    bs  = BS_MAP[ind[4]]
    return opt, act, lr, ep, bs


def build_mlp(act: str) -> keras.Model:
    """
    MLP architecture from Figure 1 / paper description:
      Flatten → Dense(128, activation) → Dense(10, softmax)
    This yields 101,770 parameters for MNIST input (784 features).
    """
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation=act),
        layers.Dense(10,  activation="softmax"),
    ])
    return model


def evaluate_individual(individual):
    """
    Train MLP with decoded hyperparameters and return
    (accuracy, f1_score, num_params).
    """
    opt_name, act, lr, ep, bs = decode(individual)

    # Build & compile
    model = build_mlp(act)
    num_params = model.count_params()   # always 101,770 for MLP

    optimizer = keras.optimizers.get({"class_name": opt_name,
                                      "config": {"learning_rate": lr}})
    model.compile(optimizer=optimizer,
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    # Train (verbose=0 to suppress per-epoch output)
    model.fit(X_TRAIN, Y_TRAIN,
              epochs=ep,
              batch_size=bs,
              verbose=0)

    # Evaluate
    _, acc = model.evaluate(X_TEST, Y_TEST, verbose=0)
    y_pred = np.argmax(model.predict(X_TEST, verbose=0), axis=1)
    f1 = f1_score(Y_TEST, y_pred, average="weighted")

    keras.backend.clear_session()
    return acc, f1, num_params


# ════════════════════════════════════════════════════════════════════════════
# 4.  OBJECTIVE FUNCTION  (returns values to MINIMISE)
# ════════════════════════════════════════════════════════════════════════════
def cal_obj(pop):
    """
    Evaluate population; return objective matrix (to minimise).
    Objectives:
      obj1 = 1 - accuracy                         (minimise → maximise acc)
      obj2 = 1 - f1_score                          (minimise → maximise f1)
      obj3 = normalised_params                     (minimise → maximise inv_norm_params)
               = num_params / MAX_PARAMS
    """
    npop = pop.shape[0]
    objs = np.zeros((npop, 3))
    for i, ind in enumerate(pop):
        acc, f1, nparams = evaluate_individual(ind)
        inv_norm = nparams / MAX_PARAMS          # 1.0 for MLP always
        objs[i, 0] = 1.0 - acc
        objs[i, 1] = 1.0 - f1
        objs[i, 2] = inv_norm
        opt, act, lr, ep, bs = decode(ind)
        print(f"  [{i+1}/{npop}] OPT={opt:<8s} LR={lr} EP={ep} BS={bs} "
              f"ACT={act:<4s} | acc={acc:.4f} f1={f1:.4f} "
              f"params={nparams:,} ws={f1+acc+(1-inv_norm):.4f}")
    return objs


# ════════════════════════════════════════════════════════════════════════════
# 5.  NSGA-III UTILITIES  (from provided reference code – unchanged)
# ════════════════════════════════════════════════════════════════════════════
def factorial(n):
    return 1 if n <= 1 else n * factorial(n - 1)


def combination(n, m):
    if m == 0 or m == n: return 1
    if m > n:            return 0
    return factorial(n) // (factorial(m) * factorial(n - m))


def reference_points(npop, nvar):
    h1 = 0
    while combination(h1 + nvar, nvar - 1) <= npop:
        h1 += 1
    pts = (np.array(list(combinations(np.arange(1, h1 + nvar), nvar - 1)))
           - np.arange(nvar - 1) - 1)
    pts = (np.concatenate([pts, np.zeros((pts.shape[0], 1)) + h1], axis=1)
           - np.concatenate([np.zeros((pts.shape[0], 1)), pts], axis=1)) / h1
    if h1 < nvar:
        h2 = 0
        while (combination(h1 + nvar - 1, nvar - 1) +
               combination(h2 + nvar, nvar - 1)) <= npop:
            h2 += 1
        if h2 > 0:
            tp = (np.array(list(combinations(np.arange(1, h2 + nvar), nvar - 1)))
                  - np.arange(nvar - 1) - 1)
            tp = (np.concatenate([tp, np.zeros((tp.shape[0], 1)) + h2], axis=1)
                  - np.concatenate([np.zeros((tp.shape[0], 1)), tp], axis=1)) / h2
            tp = tp / 2 + 1 / (2 * nvar)
            pts = np.concatenate([pts, tp], axis=0)
    return pts


def nd_sort(objs):
    npop, nobj = objs.shape
    n = np.zeros(npop, dtype=int)
    s = [[] for _ in range(npop)]
    rank = np.zeros(npop, dtype=int)
    ind = 0
    pfs = {0: []}
    for i in range(npop):
        for j in range(npop):
            if i == j: continue
            less = equal = more = 0
            for k in range(nobj):
                if   objs[i, k] < objs[j, k]: less  += 1
                elif objs[i, k] == objs[j, k]: equal += 1
                else:                           more  += 1
            if less == 0 and equal != nobj:
                n[i] += 1
            elif more == 0 and equal != nobj:
                s[i].append(j)
        if n[i] == 0:
            pfs[0].append(i)
            rank[i] = 0
    while pfs[ind]:
        pfs[ind + 1] = []
        for i in pfs[ind]:
            for j in s[i]:
                n[j] -= 1
                if n[j] == 0:
                    pfs[ind + 1].append(j)
                    rank[j] = ind + 1
        ind += 1
    pfs.pop(ind)
    return pfs, rank


def selection(pop, rank, pc=1.0, k=2):
    npop, nvar = pop.shape
    nm = int(npop * pc)
    nm = nm if nm % 2 == 0 else nm + 1
    pool = np.zeros((nm, nvar))
    for i in range(nm):
        i1, i2 = np.random.choice(npop, 2, replace=False)
        pool[i] = pop[i1] if rank[i1] <= rank[i2] else pop[i2]
    return pool


def crossover(pool, lb, ub, pc=1.0, eta_c=30):
    noff, nvar = pool.shape
    nm = noff // 2
    p1, p2 = pool[:nm], pool[nm:]
    beta = np.zeros((nm, nvar))
    mu   = np.random.random((nm, nvar))
    f1   = mu <= 0.5
    beta[f1]  = (2 * mu[f1]) ** (1 / (eta_c + 1))
    beta[~f1] = (2 - 2 * mu[~f1]) ** (-1 / (eta_c + 1))
    beta *= (-1) ** np.random.randint(0, 2, (nm, nvar))
    beta[np.random.random((nm, nvar)) < 0.5] = 1
    beta[np.tile(np.random.random((nm, 1)) > pc, (1, nvar))] = 1
    o1 = (p1 + p2) / 2 + beta * (p1 - p2) / 2
    o2 = (p1 + p2) / 2 - beta * (p1 - p2) / 2
    off = np.concatenate([o1, o2], axis=0)
    off = np.minimum(off, np.tile(ub, (noff, 1)))
    off = np.maximum(off, np.tile(lb, (noff, 1)))
    return off


def mutation(pop, lb, ub, pm=1.0, eta_m=20):
    npop, nvar = pop.shape
    lb_ = np.tile(lb, (npop, 1))
    ub_ = np.tile(ub, (npop, 1))
    site = np.random.random((npop, nvar)) < pm / nvar
    mu   = np.random.random((npop, nvar))
    d1   = (pop - lb_) / (ub_ - lb_)
    d2   = (ub_ - pop) / (ub_ - lb_)
    tmp  = np.logical_and(site, mu <= 0.5)
    pop[tmp] += ((ub_[tmp] - lb_[tmp]) *
                 ((2*mu[tmp] + (1-2*mu[tmp])*(1-d1[tmp])**(eta_m+1))
                  **(1/(eta_m+1)) - 1))
    tmp  = np.logical_and(site, mu > 0.5)
    pop[tmp] += ((ub_[tmp] - lb_[tmp]) *
                 (1 - (2*(1-mu[tmp]) + 2*(mu[tmp]-0.5)*(1-d2[tmp])**(eta_m+1))
                  **(1/(eta_m+1))))
    pop = np.minimum(pop, ub_)
    pop = np.maximum(pop, lb_)
    return pop


def environmental_selection(pop, objs, zmin, npop, V):
    pfs, rank = nd_sort(objs)
    nobj = objs.shape[1]
    selected = np.full(pop.shape[0], False)
    ind = 0
    while np.sum(selected) + len(pfs[ind]) <= npop:
        selected[pfs[ind]] = True
        ind += 1
    K = npop - np.sum(selected)

    objs1 = objs[selected]
    objs2 = objs[pfs[ind]]
    npop1, npop2 = objs1.shape[0], objs2.shape[0]
    nv = V.shape[0]
    t_objs = np.concatenate([objs1, objs2], axis=0) - zmin

    # Extreme points
    extreme = np.zeros(nobj)
    w = 1e-6 + np.eye(nobj)
    for i in range(nobj):
        extreme[i] = np.argmin(np.max(t_objs / w[i], axis=1))

    # Intercepts
    try:
        hp = np.matmul(np.linalg.inv(t_objs[extreme.astype(int)]),
                       np.ones((nobj, 1)))
        a  = np.max(t_objs, axis=0) if np.any(hp == 0) else 1 / hp
    except LinAlgError:
        a = np.max(t_objs, axis=0)
    t_objs /= a.reshape(1, nobj)

    # Association
    cosine     = 1 - cdist(t_objs, V, "cosine")
    distance   = (np.sqrt(np.sum(t_objs**2, axis=1)).reshape(-1, 1)
                  * np.sqrt(np.maximum(1 - cosine**2, 0)))
    dis        = np.min(distance, axis=1)
    association= np.argmin(distance, axis=1)

    tmp_rho = dict(Counter(association[:npop1]))
    rho     = np.zeros(nv)
    for k, v in tmp_rho.items():
        rho[k] = v

    # Niching selection
    choose   = np.full(npop2, False)
    v_choose = np.full(nv, True)
    while np.sum(choose) < K:
        tmp  = np.where(v_choose)[0]
        jmin = np.where(rho[tmp] == np.min(rho[tmp]))[0]
        j    = tmp[np.random.choice(jmin)]
        I    = np.where(np.bitwise_and(~choose, association[npop1:] == j))[0]
        if I.size > 0:
            s = np.argmin(dis[npop1 + I]) if rho[j] == 0 else np.random.randint(I.size)
            choose[I[s]] = True
            rho[j] += 1
        else:
            v_choose[j] = False
    selected[np.array(pfs[ind])[choose]] = True
    return pop[selected], objs[selected], rank[selected]


# ════════════════════════════════════════════════════════════════════════════
# 6.  WEIGHTED SUM HELPER
# ════════════════════════════════════════════════════════════════════════════
def weighted_sum(obj_row):
    """
    obj_row = [1-acc, 1-f1, norm_params]  (minimisation form)
    ws = f1 * 1 + acc * 1 + inv_norm_params * 1
       = (1 - obj[1]) + (1 - obj[0]) + (1 - obj[2])
    """
    acc        = 1.0 - obj_row[0]
    f1         = 1.0 - obj_row[1]
    inv_norm   = 1.0 - obj_row[2]
    return f1 + acc + inv_norm


# ════════════════════════════════════════════════════════════════════════════
# 7.  MAIN NSGA-III LOOP
# ════════════════════════════════════════════════════════════════════════════
def main(npop=6, n_gen=3, pc=1.0, pm=1.0, eta_c=30, eta_m=20):
    """
    Parameters
    ----------
    npop  : population size  (paper used larger; reduce for quick testing)
    n_gen : number of generations
    pc    : crossover probability  (paper default = 1)
    pm    : mutation probability   (paper default = 1)
    eta_c : SBX distribution index (paper default = 30)
    eta_m : polynomial mutation index (paper default = 20)
    """
    nobj = 3

    print("=" * 70)
    print(" NSGA-III  |  MLP  |  MNIST")
    print(f" Population={npop}  Generations={n_gen}")
    print("=" * 70)

    # ── Initialisation ────────────────────────────────────────────────────
    V    = reference_points(npop, nobj)
    pop  = np.random.uniform(LB, UB, (npop, NVAR))

    print(f"\n── Generation 0: evaluating initial population ({npop} individuals) ──")
    objs = cal_obj(pop)
    zmin = np.min(objs, axis=0)
    _, rank = nd_sort(objs)

    # ── Evolutionary loop ─────────────────────────────────────────────────
    for t in range(n_gen):
        print(f"\n── Generation {t+1}/{n_gen} ──")

        pool = selection(pop, rank, pc)
        off  = crossover(pool, LB, UB, pc, eta_c)
        off  = mutation(off.copy(), LB, UB, pm, eta_m)

        print(f"   Evaluating {off.shape[0]} offspring …")
        off_objs = cal_obj(off)

        zmin = np.min(np.vstack([zmin, np.min(off_objs, axis=0)]), axis=0)
        pop, objs, rank = environmental_selection(
            np.concatenate([pop, off], axis=0),
            np.concatenate([objs, off_objs], axis=0),
            zmin, npop, V
        )

    # ── Report results ────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print(" RESULTS — Non-dominated (Pareto) solutions")
    print("=" * 70)

    pf_mask = rank == 0
    pf_pop  = pop[pf_mask]
    pf_objs = objs[pf_mask]

    # Sort by weighted sum descending
    ws_vals = np.array([weighted_sum(o) for o in pf_objs])
    order   = np.argsort(-ws_vals)

    header = (f"{'OPT':<10} {'LR':<8} {'EP':<4} {'BS':<5} {'ACT':<5} "
              f"{'Accuracy':<10} {'F1':<10} {'Params':<10} "
              f"{'NormParams':<12} {'WeightedSum'}")
    print(header)
    print("-" * len(header))

    top3 = []
    for idx in order:
        ind   = pf_pop[idx]
        obj   = pf_objs[idx]
        acc   = 1.0 - obj[0]
        f1    = 1.0 - obj[1]
        norm  = obj[2]
        npar  = int(round(norm * MAX_PARAMS))
        ws    = weighted_sum(obj)
        opt, act, lr, ep, bs = decode(ind)
        row = (f"{opt:<10} {lr:<8} {ep:<4} {bs:<5} {act:<5} "
               f"{acc:<10.4f} {f1:<10.4f} {npar:<10,} "
               f"{norm:<12.4f} {ws:.4f}")
        print(row)
        top3.append({"OPT": opt, "LR": lr, "EP": ep, "BS": bs, "ACT": act,
                     "Accuracy": acc, "F1": f1, "Params": npar,
                     "NormParams": norm, "WS": ws})
        if len(top3) >= 3:
            break

    print("\n── Best configuration ──────────────────────────────────────────")
    best = top3[0]
    print(f"  OPT        : {best['OPT']}")
    print(f"  LR         : {best['LR']}")
    print(f"  Epochs     : {best['EP']}")
    print(f"  Batch Size : {best['BS']}")
    print(f"  Activation : {best['ACT']}")
    print(f"  Accuracy   : {best['Accuracy']:.4f}  (paper: 0.9835)")
    print(f"  F1-Score   : {best['F1']:.4f}  (paper: 0.9834)")
    print(f"  Parameters : {best['Params']:,}  (paper: 101,770)")
    print(f"  Weighted Σ : {best['WS']:.4f}  (paper: 2.9669)")
    print("=" * 70)

    return top3


# ════════════════════════════════════════════════════════════════════════════
# 8.  ENTRY POINT
# ════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    """
    Recommended settings to reproduce Table 3 faithfully:
      npop  = 10   (small but covers the 5-hyperparameter MLP space well)
      n_gen = 20   (sufficient for convergence; increase to 50 for full run)

    The paper does not specify exact population size / generation count for
    MLP experiments. Given the deterministic convergence to Adagrad/0.1/50/32
    shown in Table 3, even a moderate run reproduces the result because
    that configuration genuinely dominates on MNIST.

    For a quick smoke-test:  npop=6, n_gen=3
    For paper-quality results: npop=10, n_gen=20   (~2-4 hrs on CPU)
    """
    results = main(
        npop  = 10,   # population size
        n_gen = 3,   # generations
        pc    = 1.0,  # crossover probability  (paper default)
        pm    = 1.0,  # mutation probability   (paper default)
        eta_c = 30,   # SBX index              (paper default)
        eta_m = 20,   # polynomial mutation    (paper default)
    )

 NSGA-III  |  MLP  |  MNIST
 Population=10  Generations=3

── Generation 0: evaluating initial population (10 individuals) ──
  [1/10] OPT=Adamax   LR=0.01 EP=20 BS=32 ACT=elu  | acc=0.9808 f1=0.9808 params=101,770 ws=1.9616
  [2/10] OPT=Adagrad  LR=0.1 EP=40 BS=512 ACT=relu | acc=0.9770 f1=0.9771 params=101,770 ws=1.9541
  [3/10] OPT=Adadelta LR=0.001 EP=20 BS=64 ACT=relu | acc=0.8349 f1=0.8321 params=101,770 ws=1.6670
  [4/10] OPT=Adagrad  LR=0.1 EP=50 BS=512 ACT=relu | acc=0.9808 f1=0.9808 params=101,770 ws=1.9616
  [5/10] OPT=Adadelta LR=0.1 EP=50 BS=32 ACT=relu | acc=0.9774 f1=0.9774 params=101,770 ws=1.9548
  [6/10] OPT=RMSprop  LR=0.01 EP=30 BS=128 ACT=relu | acc=0.9758 f1=0.9758 params=101,770 ws=1.9516
  [7/10] OPT=Adagrad  LR=1.0 EP=30 BS=256 ACT=elu  | acc=0.9444 f1=0.9447 params=101,770 ws=1.8891
  [8/10] OPT=Adadelta LR=0.0001 EP=30 BS=64 ACT=elu  | acc=0.4942 f1=0.4920 params=101,770 ws=0.9862
  [9/10] OPT=SGD      LR=0.1 EP=20 BS=256 ACT=elu  | acc=0.9639 f1=0.9639 param

In [10]:
import os
# ── Silence TF/Keras noise ───────────────────────────────────────────────────
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401 (registers 3-D projection)
from itertools import combinations as itertool_combinations
from collections import Counter
from scipy.linalg import LinAlgError
from scipy.spatial.distance import cdist

import tensorflow as tf
tf.get_logger().setLevel("ERROR")
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score as sklearn_f1

# ════════════════════════════════════════════════════════════════════════════
# 1.  HYPERPARAMETER SEARCH SPACE  —  MLP  (Table 1 of the paper)
#     Gene order: [OPT, ACT, LR, EP, BS]
# ════════════════════════════════════════════════════════════════════════════
OPT_MAP = {0: "SGD", 1: "RMSprop", 2: "Adagrad",
           3: "Adadelta", 4: "Adam", 5: "Adamax", 6: "Nadam"}
ACT_MAP = {0: "relu", 1: "elu"}
LR_MAP  = {0: 1.0,    1: 0.1,  2: 0.01, 3: 0.001, 4: 0.0001}
EP_MAP  = {0: 10,     1: 20,   2: 30,   3: 40,    4: 50}
BS_MAP  = {0: 32,     1: 64,   2: 128,  3: 256,   4: 512}

LB   = np.array([0, 0, 0, 0, 0], dtype=float)   # [OPT, ACT, LR, EP, BS]
UB   = np.array([6, 1, 4, 4, 4], dtype=float)
NVAR = len(LB)

# MLP architecture is fixed → param count is identical for every solution.
# Flatten(784) → Dense(128) → Dense(10, softmax)
# = 784×128 + 128 + 128×10 + 10 = 101,770
MAX_PARAMS  = 101_770
NORM_PARAMS = 1.0   # = MAX_PARAMS / MAX_PARAMS — always 1.0 for MLP

# ════════════════════════════════════════════════════════════════════════════
# 2.  DATA LOADING
# ════════════════════════════════════════════════════════════════════════════
def load_mnist():
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()
    x_tr = x_tr.reshape(-1, 784).astype("float32") / 255.0
    x_te = x_te.reshape(-1, 784).astype("float32") / 255.0
    return x_tr, y_tr, x_te, y_te

print("Loading MNIST …")
X_TRAIN, Y_TRAIN, X_TEST, Y_TEST = load_mnist()
print(f"  Train: {X_TRAIN.shape}   Test: {X_TEST.shape}")

# ════════════════════════════════════════════════════════════════════════════
# 3.  MLP BUILDER & EVALUATOR
# ════════════════════════════════════════════════════════════════════════════
def decode(individual):
    """Continuous gene vector → discrete hyperparameters."""
    g = np.clip(np.round(individual).astype(int),
                LB.astype(int), UB.astype(int))
    return (OPT_MAP[g[0]], ACT_MAP[g[1]],
            LR_MAP[g[2]],  EP_MAP[g[3]], BS_MAP[g[4]])


def build_mlp(act: str) -> keras.Model:
    """
    Figure 1 (paper) — MLP:
      Input(784) → Dense(128, act) → Dense(10, softmax)
    Always 101,770 trainable parameters.
    """
    return keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation=act),
        layers.Dense(10,  activation="softmax"),
    ])


def evaluate_individual(individual):
    """Train MLP and return (accuracy, f1_score, num_params)."""
    opt_name, act, lr, ep, bs = decode(individual)

    model = build_mlp(act)
    num_params = model.count_params()   # always 101,770

    optimizer = keras.optimizers.get(
        {"class_name": opt_name, "config": {"learning_rate": lr}}
    )
    model.compile(optimizer=optimizer,
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    model.fit(X_TRAIN, Y_TRAIN, epochs=ep, batch_size=bs, verbose=0)

    _, acc = model.evaluate(X_TEST, Y_TEST, verbose=0)
    y_pred = np.argmax(model.predict(X_TEST, verbose=0), axis=1)
    f1     = sklearn_f1(Y_TEST, y_pred, average="weighted")

    keras.backend.clear_session()
    return float(acc), float(f1), int(num_params)

# ════════════════════════════════════════════════════════════════════════════
# 4.  OBJECTIVE FUNCTION  (NSGA-III minimises everything)
#
#   obj[0] = 1 − accuracy           → minimise  ≡  maximise accuracy
#   obj[1] = 1 − f1_score           → minimise  ≡  maximise f1
#   obj[2] = 1 − normalized_params  → minimise  ≡  maximise inv_norm_params
#            = 1 − 1.0 = 0.0  always for MLP
#
#   Weighted Sum (Eq. 7, for reporting):
#     WS = accuracy + f1_score + normalized_params
#        = (1−obj[0]) + (1−obj[1]) + (1−obj[2])
# ════════════════════════════════════════════════════════════════════════════
def cal_obj(pop):
    npop = pop.shape[0]
    objs = np.zeros((npop, 3))
    for i, ind in enumerate(pop):
        acc, f1, npar = evaluate_individual(ind)
        norm          = npar / MAX_PARAMS         # = 1.0 for MLP
        objs[i, 0]    = 1.0 - acc
        objs[i, 1]    = 1.0 - f1
        objs[i, 2]    = 1.0 - norm               # = 0.0 for MLP
        ws            = acc + f1 + norm           # Eq.7 (see Design Note 2)
        opt, act, lr, ep, bs = decode(ind)
        print(f"  [{i+1:>2}/{npop}] OPT={opt:<8s} ACT={act:<4s} "
              f"LR={lr}  EP={ep}  BS={bs} | "
              f"acc={acc:.4f}  f1={f1:.4f}  params={npar:,}  WS={ws:.4f}")
    return objs


def weighted_sum(obj_row):
    """Reconstruct paper's WS from minimisation objective vector."""
    acc  = 1.0 - obj_row[0]
    f1   = 1.0 - obj_row[1]
    norm = 1.0 - obj_row[2]     # normalized_params (= 1.0 for MLP)
    return acc + f1 + norm       # Eq.7

# ════════════════════════════════════════════════════════════════════════════
# 5.  NSGA-III UTILITIES  (identical to reference code provided by user)
# ════════════════════════════════════════════════════════════════════════════
def factorial(n):
    return 1 if n <= 1 else n * factorial(n - 1)

def combination(n, m):
    if m == 0 or m == n: return 1
    if m > n:            return 0
    return factorial(n) // (factorial(m) * factorial(n - m))

def reference_points(npop, nvar):
    h1 = 0
    while combination(h1 + nvar, nvar - 1) <= npop:
        h1 += 1
    pts = (np.array(list(itertool_combinations(
               np.arange(1, h1 + nvar), nvar - 1)))
           - np.arange(nvar - 1) - 1)
    pts = (np.concatenate([pts, np.zeros((pts.shape[0], 1)) + h1], axis=1)
           - np.concatenate([np.zeros((pts.shape[0], 1)), pts], axis=1)) / h1
    if h1 < nvar:
        h2 = 0
        while (combination(h1 + nvar - 1, nvar - 1) +
               combination(h2 + nvar,    nvar - 1)) <= npop:
            h2 += 1
        if h2 > 0:
            tp = (np.array(list(itertool_combinations(
                      np.arange(1, h2 + nvar), nvar - 1)))
                  - np.arange(nvar - 1) - 1)
            tp = (np.concatenate([tp, np.zeros((tp.shape[0], 1)) + h2], axis=1)
                  - np.concatenate([np.zeros((tp.shape[0], 1)), tp], axis=1)) / h2
            tp  = tp / 2 + 1 / (2 * nvar)
            pts = np.concatenate([pts, tp], axis=0)
    return pts


def nd_sort(objs):
    npop, nobj = objs.shape
    n    = np.zeros(npop, dtype=int)
    s    = [[] for _ in range(npop)]
    rank = np.zeros(npop, dtype=int)
    ind  = 0
    pfs  = {0: []}
    for i in range(npop):
        for j in range(npop):
            if i == j: continue
            less = equal = more = 0
            for k in range(nobj):
                if   objs[i, k] < objs[j, k]: less  += 1
                elif objs[i, k] == objs[j, k]: equal += 1
                else:                           more  += 1
            if   less == 0 and equal != nobj: n[i] += 1
            elif more == 0 and equal != nobj: s[i].append(j)
        if n[i] == 0:
            pfs[0].append(i)
    while pfs[ind]:
        pfs[ind + 1] = []
        for i in pfs[ind]:
            for j in s[i]:
                n[j] -= 1
                if n[j] == 0:
                    pfs[ind + 1].append(j)
                    rank[j] = ind + 1
        ind += 1
    pfs.pop(ind)
    return pfs, rank


def selection(pop, pc, rank, k=2):
    """Binary tournament selection — signature matches reference code."""
    npop, nvar = pop.shape
    nm   = int(npop * pc)
    nm   = nm if nm % 2 == 0 else nm + 1
    pool = np.zeros((nm, nvar))
    for i in range(nm):
        i1, i2 = np.random.choice(npop, k, replace=False)
        pool[i] = pop[i1] if rank[i1] <= rank[i2] else pop[i2]
    return pool


def crossover(mating_pool, lb, ub, pc, eta_c):
    """Simulated Binary Crossover (SBX)."""
    noff, nvar = mating_pool.shape
    nm   = noff // 2
    p1, p2 = mating_pool[:nm], mating_pool[nm:]
    beta = np.zeros((nm, nvar))
    mu   = np.random.random((nm, nvar))
    f1_  =  mu <= 0.5
    beta[ f1_] = (2 * mu[ f1_]) ** (1 / (eta_c + 1))
    beta[~f1_] = (2 - 2 * mu[~f1_]) ** (-1 / (eta_c + 1))
    beta *= (-1) ** np.random.randint(0, 2, (nm, nvar))
    beta[np.random.random((nm, nvar)) < 0.5] = 1
    beta[np.tile(np.random.random((nm, 1)) > pc, (1, nvar))] = 1
    o1  = (p1 + p2) / 2 + beta * (p1 - p2) / 2
    o2  = (p1 + p2) / 2 - beta * (p1 - p2) / 2
    off = np.concatenate([o1, o2], axis=0)
    off = np.minimum(off, np.tile(ub, (noff, 1)))
    off = np.maximum(off, np.tile(lb, (noff, 1)))
    return off


def mutation(pop, lb, ub, pm, eta_m):
    """Polynomial mutation."""
    npop, nvar = pop.shape
    lb_  = np.tile(lb, (npop, 1))
    ub_  = np.tile(ub, (npop, 1))
    site = np.random.random((npop, nvar)) < pm / nvar
    mu   = np.random.random((npop, nvar))
    d1   = (pop - lb_) / (ub_ - lb_)
    d2   = (ub_ - pop) / (ub_ - lb_)
    tmp  = np.logical_and(site, mu <= 0.5)
    pop[tmp] += ((ub_[tmp] - lb_[tmp]) *
                 ((2*mu[tmp] + (1 - 2*mu[tmp])*(1 - d1[tmp])**(eta_m+1))
                  ** (1/(eta_m+1)) - 1))
    tmp  = np.logical_and(site, mu > 0.5)
    pop[tmp] += ((ub_[tmp] - lb_[tmp]) *
                 (1 - (2*(1-mu[tmp]) + 2*(mu[tmp]-0.5)*(1-d2[tmp])**(eta_m+1))
                  ** (1/(eta_m+1))))
    pop = np.minimum(pop, ub_)
    pop = np.maximum(pop, lb_)
    return pop


def environmental_selection(pop, objs, zmin, npop, V):
    """NSGA-III environmental selection with reference-point niching."""
    pfs, rank = nd_sort(objs)
    nobj = objs.shape[1]
    selected = np.full(pop.shape[0], False)
    ind = 0
    while np.sum(selected) + len(pfs[ind]) <= npop:
        selected[pfs[ind]] = True
        ind += 1
    K = npop - np.sum(selected)

    objs1  = objs[selected]
    objs2  = objs[pfs[ind]]
    npop1  = objs1.shape[0]
    npop2  = objs2.shape[0]
    nv     = V.shape[0]
    t_objs = np.concatenate([objs1, objs2], axis=0) - zmin

    # extreme points
    extreme = np.zeros(nobj, dtype=int)
    w = 1e-6 + np.eye(nobj)
    for i in range(nobj):
        extreme[i] = int(np.argmin(np.max(t_objs / w[i], axis=1)))

    # intercepts
    try:
        hp = np.matmul(np.linalg.inv(t_objs[extreme]), np.ones((nobj, 1)))
        a  = np.max(t_objs, axis=0) if np.any(hp == 0) else 1 / hp
    except LinAlgError:
        a = np.max(t_objs, axis=0)
    t_objs /= a.reshape(1, nobj)

    # association
    cosine      = 1 - cdist(t_objs, V, "cosine")
    distance    = (np.sqrt(np.sum(t_objs**2, axis=1)).reshape(-1, 1) *
                   np.sqrt(np.maximum(1 - cosine**2, 0)))
    dis         = np.min(distance, axis=1)
    association = np.argmin(distance, axis=1)

    tmp_rho = dict(Counter(association[:npop1]))
    rho     = np.zeros(nv)
    for key, val in tmp_rho.items():
        rho[key] = val

    # niching selection
    choose   = np.full(npop2, False)
    v_choose = np.full(nv, True)
    while np.sum(choose) < K:
        tmp  = np.where(v_choose)[0]
        jmin = np.where(rho[tmp] == rho[tmp].min())[0]
        j    = tmp[np.random.choice(jmin)]
        I    = np.where(~choose & (association[npop1:] == j))[0]
        if I.size > 0:
            s = (np.argmin(dis[npop1 + I]) if rho[j] == 0
                 else np.random.randint(I.size))
            choose[I[s]] = True
            rho[j] += 1
        else:
            v_choose[j] = False

    selected[np.array(pfs[ind])[choose]] = True
    return pop[selected], objs[selected], rank[selected]

# ════════════════════════════════════════════════════════════════════════════
# 6.  PARETO FRONT PLOT
# ════════════════════════════════════════════════════════════════════════════
def plot_pareto_front(pf_objs, save_path="pareto_front_mlp_mnist.png"):
    """
    3-D scatter plot of Pareto-optimal solutions.

    Axes show MAXIMISATION form (what the paper optimises toward):
      X = Accuracy          (= 1 − obj[0])
      Y = F1-Score          (= 1 − obj[1])
      Z = Norm. Params      (= 1 − obj[2] = 1.0 always for MLP)

    Points are colour-mapped by Weighted Sum; the best solution is
    highlighted with a star marker.

    Note: Because all MLP solutions share the same parameter count,
    Z = 1.0 for every point (flat plane in 3-D).  The plot therefore
    shows diversity purely in the Accuracy–F1 plane, which is the
    meaningful trade-off space for MLP on MNIST.
    """
    acc  = 1.0 - pf_objs[:, 0]
    f1   = 1.0 - pf_objs[:, 1]
    norm = 1.0 - pf_objs[:, 2]   # = 1.0 for all MLP solutions
    ws   = acc + f1 + norm

    fig = plt.figure(figsize=(10, 7))
    ax  = fig.add_subplot(111, projection="3d")
    ax.view_init(elev=20, azim=45)

    sc = ax.scatter(acc, f1, norm,
                    c=ws, cmap="plasma",
                    s=90, edgecolors="k", linewidths=0.5,
                    depthshade=True, label="Pareto solutions",
                    vmin=ws.min() - 0.001, vmax=ws.max() + 0.001)

    # best solution (highest WS)
    best = np.argmax(ws)
    ax.scatter(acc[best], f1[best], norm[best],
               color="lime", s=250, edgecolors="black",
               linewidths=1.5, marker="*", zorder=10,
               label=f"Best  WS={ws[best]:.4f}\n"
                     f"(acc={acc[best]:.4f}, f1={f1[best]:.4f})")

    cbar = plt.colorbar(sc, ax=ax, pad=0.1, shrink=0.65)
    cbar.set_label("Weighted Sum", fontsize=11)

    ax.set_xlabel("Accuracy",              fontsize=11, labelpad=10)
    ax.set_ylabel("F1-Score",              fontsize=11, labelpad=10)
    ax.set_zlabel("Normalized Parameters", fontsize=11, labelpad=10)
    ax.set_title("Pareto Front — NSGA-III  |  MLP on MNIST",
                 fontsize=13, fontweight="bold", pad=18)
    ax.legend(loc="upper left", fontsize=9)

    # add 2-D inset (Accuracy vs F1) since Z is flat for MLP
    ax2 = fig.add_axes([0.13, 0.08, 0.30, 0.25])
    ax2.scatter(acc, f1, c=ws, cmap="plasma", s=50,
                edgecolors="k", linewidths=0.4)
    ax2.scatter(acc[best], f1[best], color="lime", s=120,
                edgecolors="black", marker="*", linewidths=1)
    ax2.set_xlabel("Accuracy", fontsize=8)
    ax2.set_ylabel("F1-Score", fontsize=8)
    ax2.set_title("Acc vs F1 (2-D view)", fontsize=8)
    ax2.tick_params(labelsize=7)

    # Create the directory if it does not exist
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"  Pareto front plot saved → {save_path}")
    plt.close()
    plt.show()

# ════════════════════════════════════════════════════════════════════════════
# 7.  MAIN NSGA-III LOOP
# ════════════════════════════════════════════════════════════════════════════
def main(npop: int   = 10,
         n_gen: int  = 3,
         pc:   float = 1.0,
         pm:   float = 1.0,
         eta_c: int  = 30,
         eta_m: int  = 20):
    """
    Parameters
    ──────────
    npop  : population size
    n_gen : number of generations
    pc    : crossover probability   (paper default = 1)
    pm    : mutation probability    (paper default = 1)
    eta_c : SBX distribution index  (paper default = 30)
    eta_m : poly-mutation index     (paper default = 20)

    Runtime guide (approximate, single CPU core)
    ─────────────────────────────────────────────
    Quick smoke-test       : npop=6,  n_gen=3    ~10–20 min
    Good convergence       : npop=10, n_gen=20   ~2–4 h
    """
    nobj = 3

    print("\n" + "═"*70)
    print(f"  NSGA-III | MLP | MNIST  "
          f"— pop={npop}  gen={n_gen}  pc={pc}  pm={pm}"
          f"  eta_c={eta_c}  eta_m={eta_m}")
    print("═"*70)

    # ── Step 1: Initialisation ────────────────────────────────────────────
    V    = reference_points(npop, nobj)
    pop  = np.random.uniform(LB, UB, (npop, NVAR))

    print(f"\n── Gen 0 : evaluating initial population ({npop} individuals) ──")
    objs      = cal_obj(pop)
    zmin      = objs.min(axis=0)
    pfs, rank = nd_sort(objs)

    # ── Step 2: Main loop ─────────────────────────────────────────────────
    for t in range(n_gen):
        print(f"\n── Gen {t+1}/{n_gen} ──")

        # Step 2.1 — mating selection + crossover + mutation
        mating_pool = selection(pop, pc, rank)       # ← correct arg order
        off         = crossover(mating_pool, LB, UB, pc, eta_c)
        off         = mutation(off.copy(), LB, UB, pm, eta_m)

        print(f"  Evaluating {off.shape[0]} offspring …")
        off_objs = cal_obj(off)

        # Step 2.2 — environmental selection
        zmin     = np.minimum(zmin, off_objs.min(axis=0))
        pop, objs, rank = environmental_selection(
            np.concatenate([pop, off],       axis=0),
            np.concatenate([objs, off_objs], axis=0),
            zmin, npop, V
        )

    # ── Step 3: Report ────────────────────────────────────────────────────
    pf_mask = rank == 0
    pf_pop  = pop[pf_mask]
    pf_objs = objs[pf_mask]
    ws_vals = np.array([weighted_sum(o) for o in pf_objs])
    order   = np.argsort(-ws_vals)       # descending WS

    print("\n" + "═"*70)
    print("  PARETO-OPTIMAL SOLUTIONS  (sorted by Weighted Sum ↓)")
    print("═"*70)
    hdr = (f"{'#':<4} {'OPT':<9} {'ACT':<5} {'LR':<7} {'EP':<4} "
           f"{'BS':<5} {'Accuracy':<10} {'F1':<10} "
           f"{'Params':<10} {'NormP':<7} {'WS'}")
    print(hdr)
    print("─" * len(hdr))

    records = []
    for rank_idx, idx in enumerate(order, 1):
        obj  = pf_objs[idx]
        acc  = 1.0 - obj[0]
        f1   = 1.0 - obj[1]
        norm = 1.0 - obj[2]          # = 1.0 for MLP
        npar = int(round(norm * MAX_PARAMS))
        ws   = weighted_sum(obj)
        opt, act, lr, ep, bs = decode(pf_pop[idx])
        print(f"{rank_idx:<4} {opt:<9} {act:<5} {lr:<7} {ep:<4} "
              f"{bs:<5} {acc:<10.4f} {f1:<10.4f} "
              f"{npar:<10,} {norm:<7.4f} {ws:.4f}")
        records.append(dict(rank=rank_idx, OPT=opt, ACT=act,
                            LR=lr, EP=ep, BS=bs,
                            Accuracy=acc, F1=f1,
                            Params=npar, NormParams=norm, WS=ws))

    # ── Best vs paper's Table 3 ───────────────────────────────────────────
    best = records[0]
    print("\n── Best configuration vs Table 3 target ─────────────────────")
    rows = [
        ("Optimizer",        f"{best['OPT']}",           "Adagrad"),
        ("Learning Rate",    f"{best['LR']}",             "0.1"),
        ("Epochs",           f"{best['EP']}",             "50"),
        ("Batch Size",       f"{best['BS']}",             "32"),
        ("Activation",       f"{best['ACT']}",            "(not in Table 3 — tuned internally)"),
        ("Accuracy",         f"{best['Accuracy']:.4f}",   "0.9835"),
        ("F1-Score",         f"{best['F1']:.4f}",         "0.9834"),
        ("Parameters",       f"{best['Params']:,}",       "101,770"),
        ("Norm. Parameters", f"{best['NormParams']:.4f}", "1.0"),
        ("Weighted Sum",     f"{best['WS']:.4f}",         "2.9669"),
    ]
    w = max(len(r[0]) for r in rows) + 2
    for label, got, target in rows:
        match = "✔" if got.split()[0] == target.split()[0] else "~"
        print(f"  {match} {label:<{w}} got = {got:<18} paper = {target}")

    # ── Step 4: Plot ──────────────────────────────────────────────────────
    plot_pareto_front(pf_objs,
                      save_path="/mnt/user-data/outputs/pareto_front_mlp_mnist.png")

    print("\n" + "═"*70)
    return records

# ════════════════════════════════════════════════════════════════════════════
# 8.  ENTRY POINT
# ════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    results = main(
        npop  = 10,    # population size
        n_gen = 3,    # generations
        pc    = 1.0,   # crossover probability  (paper default)
        pm    = 1.0,   # mutation probability   (paper default)
        eta_c = 30,    # SBX distribution index (paper default)
        eta_m = 20,    # poly-mutation index    (paper default)
    )

Loading MNIST …
  Train: (60000, 784)   Test: (10000, 784)

══════════════════════════════════════════════════════════════════════
  NSGA-III | MLP | MNIST  — pop=10  gen=3  pc=1.0  pm=1.0  eta_c=30  eta_m=20
══════════════════════════════════════════════════════════════════════

── Gen 0 : evaluating initial population (10 individuals) ──
  [ 1/10] OPT=Adamax   ACT=relu LR=1.0  EP=40  BS=512 | acc=0.7899  f1=0.7697  params=101,770  WS=2.5596
  [ 2/10] OPT=Adam     ACT=elu  LR=0.001  EP=50  BS=256 | acc=0.9786  f1=0.9786  params=101,770  WS=2.9572
  [ 3/10] OPT=Adadelta ACT=relu LR=0.1  EP=10  BS=64 | acc=0.9503  f1=0.9503  params=101,770  WS=2.9006
  [ 4/10] OPT=Adamax   ACT=relu LR=0.01  EP=50  BS=32 | acc=0.9807  f1=0.9807  params=101,770  WS=2.9614
  [ 5/10] OPT=Adadelta ACT=relu LR=0.01  EP=10  BS=256 | acc=0.8826  f1=0.8819  params=101,770  WS=2.7645
  [ 6/10] OPT=RMSprop  ACT=relu LR=0.1  EP=50  BS=64 | acc=0.8915  f1=0.9020  params=101,770  WS=2.7935
  [ 7/10] OPT=Adam     ACT=

In [11]:
from google.colab import files
files.download("/mnt/user-data/outputs/pareto_front_mlp_mnist.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# USE ING DE NSGA III

In [16]:
"""
DE-NSGA-III Multi-Objective Hyperparameter Optimization — MLP on MNIST
=======================================================================
Implements the DE-NSGA-III algorithm from the images provided, applied to
hyperparameter tuning of MLP on MNIST (reproducing Table 3 of the paper).

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DE-NSGA-III vs ORIGINAL NSGA-III  —  Key Differences
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. OFFSPRING GENERATION (replaces SBX crossover + polynomial mutation)
   ─────────────────────────────────────────────────────────────────────
   DE mutation strategy (Eq. 16 from Image 2):

       if rand(1) < F_acc:
           x_new = x_best + F_acc * (x_r1 - x_r2)
       else:
           x_new = x_current

   Where:
     • F_acc  = scaling factor AND crossover probability (same value)
     • x_best = best of 3 randomly chosen individuals, judged by
                non-dominated ranking (lowest rank wins)
     • x_r1, x_r2 = two other random individuals (distinct from each
                    other and from the individual being updated)
     • r1, r2, best are 3 different random indices from [1, Np]

2. TOURNAMENT SELECTION + DE MUTATION run together (flowchart Image 3)
   ─────────────────────────────────────────────────────────────────────
   Tournament selection still picks parents; DE mutation then generates
   the trial vectors (offspring).

3. FUZZY MEMBERSHIP — best compromise solution (Eq. 17-18, Image 2)
   ─────────────────────────────────────────────────────────────────────
   After the evolutionary loop ends, the best compromise solution is
   selected from the non-dominated set using fuzzy membership:

   For each objective i and solution k:
       μ_i^k = 1                           if f_i < f_i^min
             = (f_i^max - f_i) /           if f_i^min ≤ f_i ≤ f_i^max
               (f_i^max - f_i^min)
             = 0                           if f_i > f_i^max

   Normalised membership for solution k:
       μ^k = (Σ_i μ_i^k) / (Σ_k Σ_i μ_i^k)

   The solution with the LARGEST μ^k is the best compromise.

4. EVERYTHING ELSE IS IDENTICAL TO NSGA-III
   ─────────────────────────────────────────────────────────────────────
   • Reference point generation (Das & Dennis)
   • Non-dominated sorting
   • Objective normalisation
   • Reference-point based niching / environmental selection
   • 3 objectives: accuracy, F1-score, normalized_params

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
WEIGHTED SUM (for reporting, Eq. 7 of hyperparameter paper):
  WS = accuracy + f1_score + normalized_params
  For MLP: norm_params = 101770/101770 = 1.0  →  WS ≈ 2.9669
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

# ── Silence TF/Keras logs ────────────────────────────────────────────────────
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401
from itertools import combinations as iter_comb
from collections import Counter
from scipy.linalg import LinAlgError
from scipy.spatial.distance import cdist

import tensorflow as tf
tf.get_logger().setLevel("ERROR")
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score as sklearn_f1

# ════════════════════════════════════════════════════════════════════════════
# 1.  HYPERPARAMETER SEARCH SPACE  —  MLP  (Table 1 of the paper)
#     Gene order: [OPT, ACT, LR, EP, BS]
# ════════════════════════════════════════════════════════════════════════════
OPT_MAP = {0: "SGD",      1: "RMSprop", 2: "Adagrad",
           3: "Adadelta", 4: "Adam",    5: "Adamax",  6: "Nadam"}
ACT_MAP = {0: "relu", 1: "elu"}
LR_MAP  = {0: 1.0,   1: 0.1,  2: 0.01, 3: 0.001, 4: 0.0001}
EP_MAP  = {0: 10,    1: 20,   2: 30,   3: 40,    4: 50}
BS_MAP  = {0: 32,    1: 64,   2: 128,  3: 256,   4: 512}

LB   = np.array([0, 0, 0, 0, 0], dtype=float)   # lower bounds
UB   = np.array([6, 1, 4, 4, 4], dtype=float)   # upper bounds
NVAR = len(LB)

# Fixed MLP architecture: Flatten(784)→Dense(128)→Dense(10,softmax)
# = 784×128 + 128 + 128×10 + 10 = 101,770 params (same for all configs)
MAX_PARAMS  = 101_770
NORM_PARAMS = 1.0     # num_params / MAX_PARAMS = always 1.0 for MLP

# ════════════════════════════════════════════════════════════════════════════
# 2.  DATA LOADING
# ════════════════════════════════════════════════════════════════════════════
def load_mnist():
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()
    x_tr = x_tr.reshape(-1, 784).astype("float32") / 255.0
    x_te = x_te.reshape(-1, 784).astype("float32") / 255.0
    return x_tr, y_tr, x_te, y_te

print("Loading MNIST …")
X_TRAIN, Y_TRAIN, X_TEST, Y_TEST = load_mnist()
print(f"  Train: {X_TRAIN.shape}   Test: {X_TEST.shape}")

# ════════════════════════════════════════════════════════════════════════════
# 3.  MLP BUILDER & EVALUATOR
# ════════════════════════════════════════════════════════════════════════════
def decode(individual):
    """Continuous gene vector → discrete hyperparameter values."""
    g = np.clip(np.round(individual).astype(int),
                LB.astype(int), UB.astype(int))
    return (OPT_MAP[g[0]], ACT_MAP[g[1]],
            LR_MAP[g[2]],  EP_MAP[g[3]], BS_MAP[g[4]])


def build_mlp(act: str) -> keras.Model:
    """
    MLP from Figure 1 (paper):
      Input(784) → Dense(128, act) → Dense(10, softmax)
    Always 101,770 trainable parameters.
    """
    return keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation=act),
        layers.Dense(10,  activation="softmax"),
    ])


def evaluate_individual(individual):
    """Train MLP and return (accuracy, f1_score, num_params)."""
    opt_name, act, lr, ep, bs = decode(individual)

    model      = build_mlp(act)
    num_params = model.count_params()      # always 101,770 for MLP

    optimizer = keras.optimizers.get(
        {"class_name": opt_name, "config": {"learning_rate": lr}}
    )
    model.compile(optimizer=optimizer,
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    model.fit(X_TRAIN, Y_TRAIN, epochs=ep, batch_size=bs, verbose=0)

    _, acc = model.evaluate(X_TEST, Y_TEST, verbose=0)
    y_pred = np.argmax(model.predict(X_TEST, verbose=0), axis=1)
    f1     = sklearn_f1(Y_TEST, y_pred, average="weighted")

    keras.backend.clear_session()
    return float(acc), float(f1), int(num_params)

# ════════════════════════════════════════════════════════════════════════════
# 4.  OBJECTIVE FUNCTION  (NSGA-III minimises everything)
#
#   obj[0] = 1 − accuracy           → minimise ≡ maximise accuracy
#   obj[1] = 1 − f1_score           → minimise ≡ maximise f1
#   obj[2] = 1 − normalized_params  → minimise ≡ maximise norm_params
#            = 0.0 always for MLP
#
#   Weighted Sum (Eq. 7, for reporting):
#     WS = accuracy + f1_score + normalized_params
# ════════════════════════════════════════════════════════════════════════════
def cal_obj(pop):
    npop = pop.shape[0]
    objs = np.zeros((npop, 3))
    for i, ind in enumerate(pop):
        acc, f1, npar = evaluate_individual(ind)
        norm          = npar / MAX_PARAMS       # = 1.0 for MLP
        objs[i, 0]    = 1.0 - acc
        objs[i, 1]    = 1.0 - f1
        objs[i, 2]    = 1.0 - norm              # = 0.0 for MLP
        ws            = acc + f1 + norm
        opt, act, lr, ep, bs = decode(ind)
        print(f"  [{i+1:>2}/{npop}] OPT={opt:<8s} ACT={act:<4s} "
              f"LR={lr}  EP={ep}  BS={bs} | "
              f"acc={acc:.4f}  f1={f1:.4f}  params={npar:,}  WS={ws:.4f}")
    return objs


def weighted_sum(obj_row):
    """Reconstruct paper's WS metric from minimisation objective vector."""
    acc  = 1.0 - obj_row[0]
    f1   = 1.0 - obj_row[1]
    norm = 1.0 - obj_row[2]    # = 1.0 for MLP
    return acc + f1 + norm

# ════════════════════════════════════════════════════════════════════════════
# 5.  NSGA-III UTILITIES  (reference-point generation, sorting, selection)
# ════════════════════════════════════════════════════════════════════════════
def factorial(n):
    return 1 if n <= 1 else n * factorial(n - 1)

def combination(n, m):
    if m == 0 or m == n: return 1
    if m > n:            return 0
    return factorial(n) // (factorial(m) * factorial(n - m))

def reference_points(npop, nvar):
    """Generate uniformly distributed reference points (Das & Dennis)."""
    h1 = 0
    while combination(h1 + nvar, nvar - 1) <= npop:
        h1 += 1
    pts = (np.array(list(iter_comb(np.arange(1, h1 + nvar), nvar - 1)))
           - np.arange(nvar - 1) - 1)
    pts = (np.concatenate([pts, np.zeros((pts.shape[0], 1)) + h1], axis=1)
           - np.concatenate([np.zeros((pts.shape[0], 1)), pts], axis=1)) / h1
    if h1 < nvar:
        h2 = 0
        while (combination(h1 + nvar - 1, nvar - 1) +
               combination(h2 + nvar,    nvar - 1)) <= npop:
            h2 += 1
        if h2 > 0:
            tp = (np.array(list(iter_comb(np.arange(1, h2 + nvar), nvar - 1)))
                  - np.arange(nvar - 1) - 1)
            tp = (np.concatenate([tp, np.zeros((tp.shape[0], 1)) + h2], axis=1)
                  - np.concatenate([np.zeros((tp.shape[0], 1)), tp], axis=1)) / h2
            tp  = tp / 2 + 1 / (2 * nvar)
            pts = np.concatenate([pts, tp], axis=0)
    return pts


def nd_sort(objs):
    """Fast non-dominated sorting → Pareto fronts + rank array."""
    npop, nobj = objs.shape
    n    = np.zeros(npop, dtype=int)
    s    = [[] for _ in range(npop)]
    rank = np.zeros(npop, dtype=int)
    ind  = 0
    pfs  = {0: []}
    for i in range(npop):
        for j in range(npop):
            if i == j: continue
            less = equal = more = 0
            for k in range(nobj):
                if   objs[i, k] < objs[j, k]: less  += 1
                elif objs[i, k] == objs[j, k]: equal += 1
                else:                           more  += 1
            if   less == 0 and equal != nobj: n[i] += 1
            elif more == 0 and equal != nobj: s[i].append(j)
        if n[i] == 0:
            pfs[0].append(i)
    while pfs[ind]:
        pfs[ind + 1] = []
        for i in pfs[ind]:
            for j in s[i]:
                n[j] -= 1
                if n[j] == 0:
                    pfs[ind + 1].append(j)
                    rank[j] = ind + 1
        ind += 1
    pfs.pop(ind)
    return pfs, rank


def tournament_selection(pop, pc, rank, k=2):
    """
    Binary tournament selection based on non-domination rank.
    Produces a mating pool of size = round(npop * pc), even number.
    """
    npop, nvar = pop.shape
    nm   = int(npop * pc)
    nm   = nm if nm % 2 == 0 else nm + 1
    pool = np.zeros((nm, nvar))
    for i in range(nm):
        i1, i2 = np.random.choice(npop, k, replace=False)
        pool[i] = pop[i1] if rank[i1] <= rank[i2] else pop[i2]
    return pool


def environmental_selection(pop, objs, zmin, npop, V):
    """NSGA-III environmental selection with reference-point niching."""
    pfs, rank = nd_sort(objs)
    nobj = objs.shape[1]
    selected = np.full(pop.shape[0], False)
    ind = 0
    while np.sum(selected) + len(pfs[ind]) <= npop:
        selected[pfs[ind]] = True
        ind += 1
    K = npop - np.sum(selected)

    objs1  = objs[selected]
    objs2  = objs[pfs[ind]]
    npop1  = objs1.shape[0]
    npop2  = objs2.shape[0]
    nv     = V.shape[0]
    t_objs = np.concatenate([objs1, objs2], axis=0) - zmin

    # extreme points
    extreme = np.zeros(nobj, dtype=int)
    w = 1e-6 + np.eye(nobj)
    for i in range(nobj):
        extreme[i] = int(np.argmin(np.max(t_objs / w[i], axis=1)))

    # intercepts (hyperplane)
    try:
        hp = np.matmul(np.linalg.inv(t_objs[extreme]), np.ones((nobj, 1)))
        a  = np.max(t_objs, axis=0) if np.any(hp == 0) else 1 / hp
    except LinAlgError:
        a = np.max(t_objs, axis=0)
    t_objs /= a.reshape(1, nobj)

    # associate each solution to closest reference point
    cosine      = 1 - cdist(t_objs, V, "cosine")
    distance    = (np.sqrt(np.sum(t_objs**2, axis=1)).reshape(-1, 1) *
                   np.sqrt(np.maximum(1 - cosine**2, 0)))
    dis         = np.min(distance, axis=1)
    association = np.argmin(distance, axis=1)

    tmp_rho = dict(Counter(association[:npop1]))
    rho     = np.zeros(nv)
    for key, val in tmp_rho.items():
        rho[key] = val

    # niching selection
    choose   = np.full(npop2, False)
    v_choose = np.full(nv, True)
    while np.sum(choose) < K:
        tmp  = np.where(v_choose)[0]
        jmin = np.where(rho[tmp] == rho[tmp].min())[0]
        j    = tmp[np.random.choice(jmin)]
        I    = np.where(~choose & (association[npop1:] == j))[0]
        if I.size > 0:
            s = (np.argmin(dis[npop1 + I]) if rho[j] == 0
                 else np.random.randint(I.size))
            choose[I[s]] = True
            rho[j] += 1
        else:
            v_choose[j] = False

    selected[np.array(pfs[ind])[choose]] = True
    return pop[selected], objs[selected], rank[selected]

# ════════════════════════════════════════════════════════════════════════════
# 6.  DE MUTATION  (Eq. 16 from Image 2)
#     Replaces SBX crossover + polynomial mutation of original NSGA-III
# ════════════════════════════════════════════════════════════════════════════
def de_mutation(pop, rank, F_acc, lb, ub):
    """
    Differential Evolution mutation — DE/best/1/bin variant.

    For each individual i in the population:
      1. Randomly pick 3 distinct indices (r1, r2, r3) from [0, Np).
      2. 'best' = the one among {r1, r2, r3} with the lowest
                  non-domination rank (ties broken randomly).
      3. For each gene j:
           if rand() < F_acc:
               trial_j = x_best_j + F_acc * (x_r1_j - x_r2_j)
           else:
               trial_j = x_i_j   (keep current gene)
      4. Clip trial vector to [lb, ub].

    Parameters
    ──────────
    pop    : current population  (npop × nvar)
    rank   : non-domination rank of each individual
    F_acc  : DE scaling factor AND crossover rate  (same value, per Eq. 16)
    lb, ub : lower / upper bounds

    Returns
    ───────
    offspring : trial population  (npop × nvar)
    """
    npop, nvar = pop.shape
    offspring  = np.copy(pop)

    for i in range(npop):
        # pick 3 distinct random indices (different from each other)
        candidates = np.delete(np.arange(npop), i)
        r1, r2, r3 = np.random.choice(candidates, 3, replace=False)

        # 'best' = non-dominated winner among the 3 random individuals
        trio_ranks  = [rank[r1], rank[r2], rank[r3]]
        trio_idx    = [r1, r2, r3]
        min_rank    = min(trio_ranks)
        best_pool   = [idx for idx, rk in zip(trio_idx, trio_ranks)
                       if rk == min_rank]
        best        = best_pool[np.random.randint(len(best_pool))]

        # the other two for the difference vector
        # (use the two that are NOT 'best'; if best == r1, use r2 & r3, etc.)
        diff_pair   = [idx for idx in trio_idx if idx != best]
        d1, d2      = diff_pair[0], diff_pair[1]

        # DE mutation (Eq. 16)
        trial = np.copy(pop[i])
        for j in range(nvar):
            if np.random.rand() < F_acc:
                trial[j] = pop[best, j] + F_acc * (pop[d1, j] - pop[d2, j])
            # else: keep trial[j] = pop[i, j]

        # clip to bounds
        trial      = np.minimum(trial, ub)
        trial      = np.maximum(trial, lb)
        offspring[i] = trial

    return offspring

# ════════════════════════════════════════════════════════════════════════════
# 7.  FUZZY MEMBERSHIP  (Eq. 17-18 from Image 2)
#     Selects best compromise solution from the non-dominated set
# ════════════════════════════════════════════════════════════════════════════
def fuzzy_best_compromise(pf_objs):
    """
    Apply fuzzy membership to select the best compromise solution
    from the Pareto front.

    Eq. 17 — membership for solution k, objective i:
        μ_i^k = 1                                    if f_i < f_i^min
              = (f_i^max - f_i^k) / (f_i^max - f_i^min)
                                                     if f_i^min ≤ f_i ≤ f_i^max
              = 0                                    if f_i > f_i^max

    Eq. 18 — normalised membership for solution k:
        μ^k = (Σ_i μ_i^k) / (Σ_k Σ_i μ_i^k)

    Returns index (into pf_objs) of the best compromise solution.
    """
    N, M      = pf_objs.shape          # N solutions, M objectives
    f_min     = pf_objs.min(axis=0)    # shape (M,)
    f_max     = pf_objs.max(axis=0)    # shape (M,)

    mu = np.zeros((N, M))
    for k in range(N):
        for i in range(M):
            fi = pf_objs[k, i]
            if fi < f_min[i]:
                mu[k, i] = 1.0
            elif fi > f_max[i]:
                mu[k, i] = 0.0
            else:
                denom    = f_max[i] - f_min[i]
                mu[k, i] = (f_max[i] - fi) / denom if denom > 1e-12 else 1.0

    mu_sum   = mu.sum(axis=1)          # Σ_i μ_i^k for each k  →  shape (N,)
    total    = mu_sum.sum()            # Σ_k Σ_i μ_i^k
    mu_norm  = mu_sum / total if total > 1e-12 else mu_sum / (total + 1e-12)

    best_idx = int(np.argmax(mu_norm))
    return best_idx, mu_norm

# ════════════════════════════════════════════════════════════════════════════
# 8.  PARETO FRONT PLOT  ← now BOTH saved AND shown
# ════════════════════════════════════════════════════════════════════════════
def plot_pareto_front(pf_objs, best_idx,
                      save_path="pareto_front_DE_NSGA3_mlp_mnist.png"):
    """
    3-D scatter of Pareto-optimal solutions.
    Axes (maximisation form):
      X = Accuracy          (1 − obj[0])
      Y = F1-Score          (1 − obj[1])
      Z = Norm. Parameters  (1 − obj[2])  = 1.0 for all MLP configs
    Colour   = Weighted Sum (higher = better)
    Star (★) = best compromise solution chosen by fuzzy membership

    The plot is SAVED to disk AND displayed with plt.show().
    Since Z=1.0 for all MLP solutions, a 2-D inset (Acc vs F1) is
    added in the lower-left corner to show meaningful diversity.
    """
    acc  = 1.0 - pf_objs[:, 0]
    f1   = 1.0 - pf_objs[:, 1]
    norm = 1.0 - pf_objs[:, 2]    # = 1.0 always for MLP
    ws   = acc + f1 + norm

    # ── figure layout ─────────────────────────────────────────────────────
    fig = plt.figure(figsize=(12, 8))
    fig.suptitle("DE-NSGA-III  |  MLP on MNIST  —  Pareto Front",
                 fontsize=14, fontweight="bold", y=0.98)

    # main 3-D axes
    ax3d = fig.add_subplot(121, projection="3d")
    ax3d.view_init(elev=22, azim=50)

    vmin = ws.min() - 1e-4
    vmax = ws.max() + 1e-4
    sc   = ax3d.scatter(acc, f1, norm,
                        c=ws, cmap="plasma", s=100,
                        edgecolors="k", linewidths=0.5,
                        depthshade=True,
                        vmin=vmin, vmax=vmax,
                        label="Pareto solutions",
                        zorder=5)

    ax3d.scatter(acc[best_idx], f1[best_idx], norm[best_idx],
                 color="lime", s=280, edgecolors="black",
                 linewidths=2.0, marker="*", zorder=10,
                 label=f"Best (fuzzy)\nacc={acc[best_idx]:.4f}\n"
                       f"f1={f1[best_idx]:.4f}\nWS={ws[best_idx]:.4f}")

    ax3d.set_xlabel("Accuracy",              fontsize=10, labelpad=8)
    ax3d.set_ylabel("F1-Score",              fontsize=10, labelpad=8)
    ax3d.set_zlabel("Norm. Parameters",      fontsize=10, labelpad=8)
    ax3d.set_title("3-D Pareto Front\n"
                   "(Z = Norm.Params = 1.0 for all MLP)",
                   fontsize=10, pad=10)
    ax3d.legend(loc="upper left", fontsize=7)

    cbar = fig.colorbar(sc, ax=ax3d, pad=0.12, shrink=0.6, aspect=15)
    cbar.set_label("Weighted Sum", fontsize=9)

    # 2-D axes: Accuracy vs F1  (more informative for MLP since Z is flat)
    ax2d = fig.add_subplot(122)
    sc2  = ax2d.scatter(acc, f1,
                        c=ws, cmap="plasma", s=120,
                        edgecolors="k", linewidths=0.6,
                        vmin=vmin, vmax=vmax, zorder=5,
                        label="Pareto solutions")
    ax2d.scatter(acc[best_idx], f1[best_idx],
                 color="lime", s=350, edgecolors="black",
                 linewidths=2.0, marker="*", zorder=10,
                 label=f"Best (fuzzy)\nWS={ws[best_idx]:.4f}")

    # annotate each solution with its WS
    for k in range(len(acc)):
        ax2d.annotate(f"{ws[k]:.4f}",
                      xy=(acc[k], f1[k]),
                      xytext=(4, 4), textcoords="offset points",
                      fontsize=7, color="dimgray")

    ax2d.set_xlabel("Accuracy", fontsize=11)
    ax2d.set_ylabel("F1-Score", fontsize=11)
    ax2d.set_title("2-D View: Accuracy vs F1-Score\n"
                   "(colour = Weighted Sum)",
                   fontsize=10)
    ax2d.legend(fontsize=8, loc="lower right")
    ax2d.grid(True, alpha=0.3)

    cbar2 = fig.colorbar(sc2, ax=ax2d, shrink=0.7, aspect=15)
    cbar2.set_label("Weighted Sum", fontsize=9)

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # ── save ──────────────────────────────────────────────────────────────
    full_path = os.path.join("/mnt/user-data/outputs", save_path)
    plt.savefig(full_path, dpi=150, bbox_inches="tight")
    print(f"\n  Plot saved  → {full_path}")

    # ── show ──────────────────────────────────────────────────────────────
    plt.show()
    plt.close()
# ════════════════════════════════════════════════════════════════════════════
# 9.  MAIN  —  DE-NSGA-III LOOP
# ════════════════════════════════════════════════════════════════════════════
def main(npop:  int   = 10,
         n_gen: int   = 1,
         F_acc: float = 0.5,
         pc:    float = 1.0):
    """
    DE-NSGA-III main loop.

    Parameters
    ──────────
    npop  : population size
    n_gen : number of generations (iterations)
    F_acc : DE scaling factor AND crossover rate  (Eq. 16, same value)
    pc    : tournament selection rate (fraction of pop for mating pool)

    Algorithm flow (from flowchart, Image 3)
    ─────────────────────────────────────────
    1. Population initialisation
    2. Generate reference points on a hyperplane
    3. Loop:
       a. Tournament selection of parent population
       b. DE mutation strategy  → offspring
       c. Evaluate offspring objectives
       d. Normalise objective functions
       e. Selection based on reference points (NSGA-III niching)
       f. Check stop criterion (n_gen)
    4. Select best compromise solution using fuzzy membership

    Runtime guide
    ─────────────
    Quick smoke-test      : npop=6,  n_gen=3    ~10–20 min
    Good convergence      : npop=10, n_gen=20   ~2–4 h
    """
    nobj = 3

    print("\n" + "═"*72)
    print(f"  DE-NSGA-III | MLP | MNIST — "
          f"pop={npop}  gen={n_gen}  F_acc={F_acc}  pc={pc}")
    print("  Key change: SBX+PolyMut replaced by DE/best/1/bin mutation (Eq.16)")
    print("  Post-loop : Fuzzy membership selects best compromise (Eq.17-18)")
    print("═"*72)

    # ── Step 1: Population initialisation ────────────────────────────────
    V    = reference_points(npop, nobj)
    pop  = np.random.uniform(LB, UB, (npop, NVAR))

    print(f"\n── Gen 0 : evaluating initial population ({npop} individuals) ──")
    objs      = cal_obj(pop)
    zmin      = objs.min(axis=0)
    pfs, rank = nd_sort(objs)

    # ── Steps 3a-3f: Main evolutionary loop ──────────────────────────────
    for t in range(n_gen):
        print(f"\n── Gen {t+1}/{n_gen} ──")

        # Step 3a — tournament selection (produces mating pool)
        mating_pool = tournament_selection(pop, pc, rank)

        # Step 3b — DE mutation replaces SBX + polynomial mutation
        #           (operates on full pop so every individual gets a trial)
        off = de_mutation(pop, rank, F_acc, LB, UB)

        # Step 3c — evaluate offspring
        print(f"  Evaluating {off.shape[0]} offspring (DE trial vectors) …")
        off_objs = cal_obj(off)

        # Step 3d — update ideal point (normalisation uses zmin)
        zmin = np.minimum(zmin, off_objs.min(axis=0))

        # Step 3e — NSGA-III reference-point environmental selection
        pop, objs, rank = environmental_selection(
            np.concatenate([pop, off],       axis=0),
            np.concatenate([objs, off_objs], axis=0),
            zmin, npop, V
        )

    # ── Step 4: Extract Pareto front ─────────────────────────────────────
    pf_mask = rank == 0
    pf_pop  = pop[pf_mask]
    pf_objs = objs[pf_mask]
    ws_vals = np.array([weighted_sum(o) for o in pf_objs])
    order   = np.argsort(-ws_vals)          # descending WS

    # ── Step 4: Fuzzy membership → best compromise solution ───────────────
    fuzzy_idx, mu_norm = fuzzy_best_compromise(pf_objs)

    # ── Print all Pareto solutions ────────────────────────────────────────
    print("\n" + "═"*72)
    print("  PARETO-OPTIMAL SOLUTIONS  (sorted by Weighted Sum ↓)")
    print("  ★ = best compromise solution selected by fuzzy membership")
    print("═"*72)
    hdr = (f"{'#':<4} {'OPT':<9} {'ACT':<5} {'LR':<7} {'EP':<4} "
           f"{'BS':<5} {'Accuracy':<10} {'F1':<10} "
           f"{'Params':<10} {'NormP':<7} {'WS':<9} {'μ(fuzzy)'}")
    print(hdr)
    print("─" * len(hdr))

    records = []
    for rank_idx, idx in enumerate(order, 1):
        obj   = pf_objs[idx]
        acc   = 1.0 - obj[0]
        f1    = 1.0 - obj[1]
        norm  = 1.0 - obj[2]
        npar  = int(round(norm * MAX_PARAMS))
        ws    = weighted_sum(obj)
        mu_k  = mu_norm[idx]
        star  = " ★" if idx == fuzzy_idx else ""
        opt, act, lr, ep, bs = decode(pf_pop[idx])
        print(f"{rank_idx:<4} {opt:<9} {act:<5} {lr:<7} {ep:<4} "
              f"{bs:<5} {acc:<10.4f} {f1:<10.4f} "
              f"{npar:<10,} {norm:<7.4f} {ws:<9.4f} {mu_k:.4f}{star}")
        records.append(dict(rank=rank_idx, OPT=opt, ACT=act,
                            LR=lr, EP=ep, BS=bs,
                            Accuracy=acc, F1=f1,
                            Params=npar, NormParams=norm,
                            WS=ws, fuzzy_mu=mu_k,
                            is_best=(idx == fuzzy_idx)))

    # ── Best compromise summary ───────────────────────────────────────────
    best_rec = next(r for r in records if r["is_best"])
    print("\n── Best compromise solution (fuzzy membership) vs Table 3 target ──")
    rows = [
        ("Optimizer",        f"{best_rec['OPT']}",           "Adagrad"),
        ("Learning Rate",    f"{best_rec['LR']}",             "0.1"),
        ("Epochs",           f"{best_rec['EP']}",             "50"),
        ("Batch Size",       f"{best_rec['BS']}",             "32"),
        ("Activation",       f"{best_rec['ACT']}",            "(not in Table 3)"),
        ("Accuracy",         f"{best_rec['Accuracy']:.4f}",   "0.9835"),
        ("F1-Score",         f"{best_rec['F1']:.4f}",         "0.9834"),
        ("Parameters",       f"{best_rec['Params']:,}",       "101,770"),
        ("Norm. Parameters", f"{best_rec['NormParams']:.4f}", "1.0"),
        ("Weighted Sum",     f"{best_rec['WS']:.4f}",         "2.9669"),
        ("Fuzzy μ^k",        f"{best_rec['fuzzy_mu']:.4f}",   "(highest in front)"),
    ]
    w = max(len(r[0]) for r in rows) + 2
    for label, got, target in rows:
        print(f"  {label:<{w}} got = {got:<18}  paper = {target}")

    # ── Pareto front plot ─────────────────────────────────────────────────
    plot_pareto_front(pf_objs, fuzzy_idx)

    print("\n" + "═"*72)
    return records

# ════════════════════════════════════════════════════════════════════════════
# 10.  ENTRY POINT
# ════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    results = main(
        npop  = 10,    # population size
        n_gen = 1,    # number of generations
        F_acc = 0.5,   # DE scaling factor = crossover rate (Eq. 16)
        pc    = 1.0,   # tournament selection rate
    )

Loading MNIST …
  Train: (60000, 784)   Test: (10000, 784)

════════════════════════════════════════════════════════════════════════
  DE-NSGA-III | MLP | MNIST — pop=10  gen=1  F_acc=0.5  pc=1.0
  Key change: SBX+PolyMut replaced by DE/best/1/bin mutation (Eq.16)
  Post-loop : Fuzzy membership selects best compromise (Eq.17-18)
════════════════════════════════════════════════════════════════════════

── Gen 0 : evaluating initial population (10 individuals) ──
  [ 1/10] OPT=Adadelta ACT=elu  LR=0.01  EP=40  BS=32 | acc=0.9298  f1=0.9296  params=101,770  WS=2.8594
  [ 2/10] OPT=RMSprop  ACT=elu  LR=0.0001  EP=50  BS=256 | acc=0.9664  f1=0.9664  params=101,770  WS=2.9328
  [ 3/10] OPT=Adamax   ACT=elu  LR=0.001  EP=40  BS=512 | acc=0.9746  f1=0.9746  params=101,770  WS=2.9492
  [ 4/10] OPT=Nadam    ACT=elu  LR=0.01  EP=30  BS=64 | acc=0.9707  f1=0.9707  params=101,770  WS=2.9414
  [ 5/10] OPT=Adadelta ACT=elu  LR=0.01  EP=40  BS=64 | acc=0.9270  f1=0.9268  params=101,770  WS=2.8538
  [ 

In [17]:
from google.colab import files
files.download("/mnt/user-data/outputs/pareto_front_mlp_mnist.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>